In [37]:
!gzip -c sample_candidates.json > sample_candidates.json.gz
!ls -l sample_candidates.json.gz

-rw-r--r-- 1 root root 34064 Jul  2 09:33 sample_candidates.json.gz


In [38]:
# 1. Inspect the sizes of your files to check if they are empty
!ls -lh sample_candidates.json
!ls -lh sample_candidates.json.gz

# 2. Delete the corrupted or empty .gz archive
!rm -f sample_candidates.json.gz

# 3. Create a fresh, working compressed file from your original JSON
!gzip -c sample_candidates.json > sample_candidates.json.gz

# 4. Verify that the new file size is greater than 0
!ls -lh sample_candidates.json.gz

-rw-r--r-- 1 root root 294K Jul  2 09:33 sample_candidates.json
-rw-r--r-- 1 root root 34K Jul  2 09:33 sample_candidates.json.gz
-rw-r--r-- 1 root root 34K Jul  2 09:33 sample_candidates.json.gz


In [39]:
pip install pandas numpy rank_bm25 sentence-transformers faiss-cpu streamlit

In [40]:
%%writefile pipeline.py
import gzip
import json
import csv
import numpy as np
from datetime import datetime
from rank_bm25 import BM25Okapi
import faiss
from sentence_transformers import SentenceTransformer

# ============================================================================
# UNIVERSAL SMART LOADER
# ============================================================================
def stream_candidates(file_path):
    """
    Safely reads candidates regardless of format discrepancies:
    Handles .json, .jsonl, compressed (.gz), uncompressed, arrays, or lines.
    """
    raw_data = ""

    # Try reading as a genuine Gzip compressed file first
    try:
        with gzip.open(file_path, 'rt', encoding='utf-8') as f:
            raw_data = f.read()
    except Exception:
        # Fallback: Read as a plain uncompressed text file
        with open(file_path, 'r', encoding='utf-8') as f:
            raw_data = f.read()

    raw_data = raw_data.strip()
    if not raw_data:
        raise ValueError(f"The file at '{file_path}' is completely empty!")

    # Format Strategy A: Standard JSON Array [...]
    if raw_data.startswith('['):
        data = json.loads(raw_data)
        for candidate in data:
            yield candidate

    # Format Strategy B: JSON Lines (JSONL) where each line is its own object
    else:
        for line in raw_data.splitlines():
            if line.strip():
                yield json.loads(line)

# ============================================================================
# CORE ENGINE LAYERS
# ============================================================================
def is_valid_profile(candidate):
    return True # Keeps data inclusive for small sample sanity testing

class CandidateRankerEngine:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
        self.candidate_metadata = {}
        self.faiss_index = None
        self.corpus_for_bm25 = []
        self.candidate_ids_ordered = []

    def build_indexes(self, file_path):
        embeddings_list = []
        print(f"Starting parsing stream for target file: {file_path}")

        for candidate in stream_candidates(file_path):
            if not is_valid_profile(candidate):
                continue

            cid = candidate.get("candidate_id")
            if not cid:
                continue

            self.candidate_metadata[cid] = candidate
            self.candidate_ids_ordered.append(cid)

            profile = candidate.get("profile", {})
            skills_text = ", ".join([s.get("name", "") for s in candidate.get("skills", [])])
            resume_context = f"{profile.get('headline', '')} {profile.get('summary', '')} Skills: {skills_text}"

            self.corpus_for_bm25.append(resume_context.lower().split())
            vec = self.model.encode(resume_context)
            embeddings_list.append(vec)

        if len(embeddings_list) == 0:
            raise ValueError("🔴 ERROR: Zero records loaded successfully. Check file configuration context.")

        embeddings_matrix = np.atleast_2d(np.array(embeddings_list)).astype('float32')
        faiss.normalize_L2(embeddings_matrix)

        self.faiss_index = faiss.IndexFlatIP(embeddings_matrix.shape[1])
        self.faiss_index.add(embeddings_matrix)
        self.bm25 = BM25Okapi(self.corpus_for_bm25)
        print(f"✅ Setup Complete! Cleanly indexed {len(self.candidate_ids_ordered)} profile pools.")

    def run_ranking_pipeline(self, job_description, bm25_weight=0.3, semantic_weight=0.7):
        jd_vector = self.model.encode([job_description]).astype('float32')
        faiss.normalize_L2(jd_vector)

        num_candidates = len(self.candidate_ids_ordered)
        faiss_scores, faiss_indices = self.faiss_index.search(jd_vector, num_candidates)
        dense_score_map = {self.candidate_ids_ordered[idx]: score for score, idx in zip(faiss_scores[0], faiss_indices[0])}

        jd_tokens = job_description.lower().split()
        bm25_raw_scores = self.bm25.get_scores(jd_tokens)
        lexical_score_map = {self.candidate_ids_ordered[i]: score for i, score in enumerate(bm25_raw_scores)}

        final_candidates_pool = []
        for cid in self.candidate_ids_ordered:
            profile = self.candidate_metadata[cid].get("profile", {})
            dense_score = dense_score_map.get(cid, 0.0)
            lexical_score = lexical_score_map.get(cid, 0.0) / (max(bm25_raw_scores) if max(bm25_raw_scores) > 0 else 1.0)
            base_hybrid_score = (bm25_weight * lexical_score) + (semantic_weight * dense_score)

            final_candidates_pool.append({
                "candidate_id": cid,
                "score": float(base_hybrid_score),
                "reasoning": f"{profile.get('current_title', 'Engineer')} matched successfully."
            })

        sorted_pool = sorted(final_candidates_pool, key=lambda x: (-x["score"], x["candidate_id"]))
        top_100 = sorted_pool[:100]
        for rank_idx, item in enumerate(top_100, 1):
            item["rank"] = rank_idx

        return top_100

def write_submission_csv(top_100_output, output_filename="team_submission.csv"):
    fields = ["candidate_id", "rank", "score", "reasoning"]
    with open(output_filename, mode="w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for row in top_100_output:
            writer.writerow({k: row[k] for k in fields})

Overwriting pipeline.py


In [41]:
import gzip
import json

input_file = "sample_candidates.json.gz"
output_file = "fixed_candidates.json.gz"

with gzip.open(input_file, "rt", encoding="utf-8") as f_in, \
     gzip.open(output_file, "wt", encoding="utf-8") as f_out:

    for line_num, line in enumerate(f_in, 1):
        clean_line = line.strip()
        if not clean_line:
            print(f"Skipping empty line at line {line_num}")
            continue
        try:
            # Test if it is valid JSON
            json.loads(clean_line)
            f_out.write(line)
        except json.JSONDecodeError:
            print(f"Malformated JSON found and skipped at line {line_num}: {clean_line[:30]}...")


Streaming output truncated to the last 5000 lines.
Malformated JSON found and skipped at line 3243: "end_date": "2014-01-23",...
Malformated JSON found and skipped at line 3244: "duration_months": 26,...
Malformated JSON found and skipped at line 3245: "is_current": false,...
Malformated JSON found and skipped at line 3246: "industry": "IT Services",...
Malformated JSON found and skipped at line 3247: "company_size": "10001+",...
Malformated JSON found and skipped at line 3248: "description": "Business analy...
Malformated JSON found and skipped at line 3249: }...
Malformated JSON found and skipped at line 3250: ],...
Malformated JSON found and skipped at line 3251: "education": [...
Malformated JSON found and skipped at line 3252: {...
Malformated JSON found and skipped at line 3253: "institution": "Tier-3 Enginee...
Malformated JSON found and skipped at line 3254: "degree": "B.Tech",...
Malformated JSON found and skipped at line 3255: "field_of_study": "Artificial ...
Malformated JSO

In [42]:
import gzip
with gzip.open("sample_candidates.json.gz", "rt") as f:
    for i in range(5):
        print(f"Line {i+1}: {repr(f.readline())}")


Line 1: '[\n'
Line 2: '  {\n'
Line 3: '    "candidate_id": "CAND_0000001",\n'
Line 4: '    "profile": {\n'
Line 5: '      "anonymized_name": "Ira Vora",\n'


In [43]:
import importlib
import pipeline

# Force notebook environment cache to pick up our universal loader adjustments
importlib.reload(pipeline)
from pipeline import CandidateRankerEngine, write_submission_csv

# Instantiate the engine
engine = CandidateRankerEngine()

# Point directly to the original file to remove compression corruption errors
engine.build_indexes("sample_candidates.json")

# Execute Search Query Run
jd_query = "Senior AI Engineer Founding Team. Depth in ML systems, embeddings, ranking, retrieval pipelines."
top_100_results = engine.run_ranking_pipeline(jd_query, bm25_weight=0.3, semantic_weight=0.7)

# Export the submission table
write_submission_csv(top_100_results, "team_submission.csv")
print("🎉 Success! The submission file was generated without errors.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Starting parsing stream for target file: sample_candidates.json
✅ Setup Complete! Cleanly indexed 50 profile pools.
🎉 Success! The submission file was generated without errors.
